# Reinforcement Learning: Learning to Play Breakout

Reinforcement Learning: Learning to Play a Game
Iman Mossavat

Fontys ICT

September 2026


Reinforcement Learning (RL) is a way of teaching an agent to make decisions by interacting with an environment.

In this notebook, we use a **pre-trained RL agent** to play Atari Breakout. The agent has already learned how to play, so we can focus on understanding how RL works.

In [4]:
# ============================================================
# Clean Atari + Stable-Baselines3 installation
# ============================================================

!pip install -q \
    "stable-baselines3==2.9.0" \
    "gymnasium[atari]" \
    "ale-py" \
    "shimmy>=2.0,<3.0"

## The required SW packages and models

There is a whole ecosystem around reinforcement learning that makes it possible to build and reuse agents without implementing everything from scratch.

**Gymnasium** provides environments. An environment defines the observations, actions, rewards, and episode endings. You can find many standard environments there, such as CartPole, LunarLander, and Atari games.

**Stable-Baselines3 (SB3)** provides ready-made RL algorithms such as DQN, PPO, A2C, and SAC. We can use these algorithms to train agents without implementing the learning algorithms ourselves.

**Hugging Face** and other model repositories contain pretrained RL models that you can download and run. This is useful when you want to experiment with an agent without waiting for it to train.

An important idea is that a pretrained model must be compatible with the environment it was trained on. In particular, the **observation format, action space, and preprocessing need to match**. Here, the DQN was trained for `BreakoutNoFrameskip-v4`, so we create that environment and apply the same preprocessing expected by the model.

### What is the preprocessing?

The raw game does not necessarily go directly into the neural network. Images might be resized or converted to grayscale, frames might be stacked together, and other standard transformations might be applied.

For example, this agent uses **four consecutive frames** so that it can infer movement from the images. The `AtariWrapper` and `VecFrameStack` take care of this for us.

You should know **why preprocessing exists and that it must match the trained model**. You do not need to understand the implementation of these wrappers yet.

### What can you ignore for now?

You can treat `DummyVecEnv`, `VecFrameStack`, `AtariWrapper`, checkpoint compatibility settings, and the details of downloading/loading the model as library machinery.


In [ ]:
# ============================================================
# Download the pretrained model
# ============================================================

import urllib.request

url = (
    "https://huggingface.co/sb3/"
    "dqn-BreakoutNoFrameskip-v4/resolve/main/"
    "dqn-BreakoutNoFrameskip-v4.zip"
)

checkpoint = "/content/dqn-BreakoutNoFrameskip-v4.zip"

urllib.request.urlretrieve(url, checkpoint)

print("Pretrained model downloaded:")
print(checkpoint)
# ============================================================
# Create the exact environment expected by the model
# ============================================================

import gymnasium as gym

from stable_baselines3.common.atari_wrappers import AtariWrapper
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack


def make_breakout():
    # The pretrained model was trained on this exact
    # NoFrameskip Atari environment.
    env = gym.make(
        "BreakoutNoFrameskip-v4",
        render_mode="rgb_array"
    )

    # Same preprocessing used by the pretrained model.
    env = AtariWrapper(env)

    return env


# SB3 vectorized environment
env = DummyVecEnv([make_breakout])

# The pretrained model expects 4 consecutive frames.
env = VecFrameStack(env, n_stack=4)

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)
# ============================================================
# Load the PRETRAINED DQN
# ============================================================

from stable_baselines3 import DQN

model = DQN.load(
    checkpoint,
    env=env,
    custom_objects={
        # The old checkpoint was trained with this setting.
        # We are NOT training, so the replay buffer is irrelevant.
        "optimize_memory_usage": False,
        "handle_timeout_termination": False,
        "replay_buffer": None,
    }
)

print("Pretrained DQN loaded successfully.")

Pretrained model downloaded:
/content/dqn-BreakoutNoFrameskip-v4.zip
Observation space: Box(0, 255, (84, 84, 4), uint8)
Action space: Discrete(4)
Wrapping the env in a VecTransposeImage.
Pretrained DQN loaded successfully.


## Visualization (feel free to ignore)

The following functions are only used to **display the game nicely in Colab**.

They handle things such as converting the game frames into images and updating the display.

You do not need to understand how they work for now. The important part is that `play_game()` will show us what the trained agent is doing which comes next.

In [16]:
from IPython.display import display, HTML, Javascript
import base64
import io
import time
from PIL import Image


def setup_display():
    """Create the persistent game display in the notebook."""

    display(HTML("""
    <div style="text-align:center;">

        <img id="breakout_screen"
             width="420"
             style="image-rendering: pixelated;">

        <div id="breakout_info"
             style="font-family: monospace; margin-top:10px;">
            Starting game...
        </div>

    </div>

    <script>
    window.updateBreakout = function(
        imageData, step, action, reward, score
    ) {
        document.getElementById("breakout_screen").src =
            "data:image/png;base64," + imageData;

        document.getElementById("breakout_info").innerHTML =
            "Step: " + step +
            " | Action: " + action +
            " | Reward: " + reward +
            " | Score: " + score;
    };
    </script>
    """))


def update_display(frame, step, action, reward, score):
    """Convert a game frame into an image and display it."""

    image = Image.fromarray(frame)

    buffer = io.BytesIO()
    image.save(buffer, format="PNG")

    image_data = base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")

    js = Javascript(f"""
        if (window.updateBreakout) {{
            window.updateBreakout(
                "{image_data}",
                {step},
                {int(action[0])},
                {float(reward[0])},
                {float(score)}
            );
        }}
    """)

    display(js)



## Let the agent play

Now we can see the RL loop in action.

The agent repeatedly:

**observes the game → chooses an action → receives a reward and new observation**

The DQN is already trained, so it does not learn anything here. It simply uses what it has learned to choose actions.

Focus on these two lines:

```python
action, _ = model.predict(obs)

obs, reward, done, info = env.step(action)

In [18]:

def play_game(model, env, max_steps=1500, frame_delay=0.08):
    """Let a trained agent play the game."""

    setup_display()

    obs = env.reset()
    total_reward = 0

    for step in range(max_steps):

        # Agent chooses an action
        action, _ = model.predict(
            obs,
            deterministic=True
        )

        # Environment responds
        obs, reward, done, info = env.step(action)

        total_reward += reward[0]

        # Display the current game
        frame = env.render()

        update_display(
            frame,
            step,
            action,
            reward,
            total_reward
        )

        time.sleep(frame_delay)

        # Restart when the game ends
        if done[0]:
            print("Game over. Starting a new game...")

            time.sleep(1)

            obs = env.reset()
            total_reward = 0

    print("Demo finished.")

play_game(model, env)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

KeyboardInterrupt: 